In [ ]:
import os
import sys
import json
import importlib
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")
config = json.load(open(os.path.join(BASE_DIR, "config.json")))
context_config = json.load(open(os.path.join(BASE_DIR, "context_config.json")))
RESULTS_DIR = config["paths"]["results_dir"]
for fn in ("phase9_run_index.json", "phase9_selected_lambda.json", "phase9_config.json", "leakage_report.json"):
    assert os.path.exists(os.path.join(RESULTS_DIR, fn)), f"Missing results/{fn} -- run the earlier phases first."
run_index = json.load(open(os.path.join(RESULTS_DIR, "phase9_run_index.json")))
selected = json.load(open(os.path.join(RESULTS_DIR, "phase9_selected_lambda.json")))
p9cfg = json.load(open(os.path.join(RESULTS_DIR, "phase9_config.json")))
leakage = json.load(open(os.path.join(RESULTS_DIR, "leakage_report.json")))
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMOTIONS, IN_DIMS = config["emotions"], context_config["in_dims"]
C = len(EMOTIONS)
BASELINE, PROPOSED = "baseline", selected["selected_experiment"]
P10_DIR = os.path.join(RESULTS_DIR, "phase10")
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(P10_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
SEEDS = sorted({r["seed"] for r in run_index})
print(f"Device: {device} | baseline='{BASELINE}' vs proposed='{PROPOSED}' (lambda selected on VALIDATION) | seeds: {SEEDS}")

In [ ]:
MODULE_SRC = r'''
import numpy as np
import scipy.stats as st
from sklearn.metrics import roc_auc_score


def mean_std(x):
    x = np.asarray(x, dtype=np.float64)
    return float(x.mean()), (float(x.std(ddof=1)) if len(x) > 1 else float("nan"))


def paired_stats(a, b, alpha=0.05):
    """Paired comparison of a vs b over seeds (difference = a - b): mean, sample std, paired t-test p, (1-alpha) t-CI."""
    d = np.asarray(a, dtype=np.float64) - np.asarray(b, dtype=np.float64)
    n = len(d)
    out = {"n": n, "mean_diff": float(d.mean()), "std_diff": float(d.std(ddof=1)) if n > 1 else float("nan"),
           "p_value": float("nan"), "ci_low": float("nan"), "ci_high": float("nan"), "n_a_higher": int((d > 0).sum())}
    if n >= 2:
        se = d.std(ddof=1) / np.sqrt(n)
        if se == 0:
            out["p_value"] = 1.0 if d.mean() == 0 else 0.0
            out["ci_low"] = out["ci_high"] = float(d.mean())
        else:
            t = d.mean() / se
            out["p_value"] = float(2 * st.t.sf(abs(t), n - 1))
            h = st.t.ppf(1 - alpha / 2, n - 1) * se
            out["ci_low"], out["ci_high"] = float(d.mean() - h), float(d.mean() + h)
    return out


def holm(pvals):
    """Holm step-down adjusted p-values (NaN-safe: NaNs stay NaN and are ignored)."""
    p = np.asarray(pvals, dtype=np.float64)
    adj = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    idx = np.flatnonzero(ok)
    order = idx[np.argsort(p[idx])]
    m = len(order)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj


def describe_diff(s, alpha=0.05):
    """One sentence, generated from the numbers only (new - reference)."""
    if s["n"] < 2 or np.isnan(s["p_value"]):
        return f"mean difference {s['mean_diff']:+.4f} (n={s['n']} seed; no inference possible)"
    core = f"mean difference {s['mean_diff']:+.4f}, 95% CI [{s['ci_low']:+.4f}, {s['ci_high']:+.4f}], paired t-test p={s['p_value']:.3g}, n={s['n']} seeds"
    if s["ci_low"] > 0:
        return f"higher ({core})"
    if s["ci_high"] < 0:
        return f"lower ({core})"
    return f"no significant difference ({core}; the interval includes 0)"


def confusion_flat(y, p, C):
    return np.bincount(np.asarray(y) * C + np.asarray(p), minlength=C * C).astype(np.float64)


def f1_per_class_from_conf(conf_flat, C):
    conf = np.asarray(conf_flat, dtype=np.float64).reshape(C, C)       
    tp, row, col = np.diag(conf), conf.sum(1), conf.sum(0)
    return np.where(row + col > 0, 2 * tp / np.maximum(row + col, 1e-300), 0.0)


def macro_f1_from_conf(conf_flat, C):
    return float(f1_per_class_from_conf(conf_flat, C).mean())


def dialogue_bootstrap_diff(y, p_new, p_ref, dialogue_ids, C, n_boot=1000, seed=0, alpha=0.05):
    """Paired dialogue-level bootstrap of macro-F1(new) - macro-F1(ref) on one test set."""
    y, p_new, p_ref = np.asarray(y), np.asarray(p_new), np.asarray(p_ref)
    ids, inv = np.unique(dialogue_ids, return_inverse=True)
    D = len(ids)
    cn, cr = np.zeros((D, C * C)), np.zeros((D, C * C))
    np.add.at(cn, (inv, y * C + p_new), 1)
    np.add.at(cr, (inv, y * C + p_ref), 1)
    rng = np.random.RandomState(seed)
    diffs = np.empty(n_boot)
    for b in range(n_boot):
        cnt = np.bincount(rng.randint(0, D, D), minlength=D).astype(np.float64)
        diffs[b] = macro_f1_from_conf(cnt @ cn, C) - macro_f1_from_conf(cnt @ cr, C)
    point = macro_f1_from_conf(cn.sum(0), C) - macro_f1_from_conf(cr.sum(0), C)
    lo, hi = np.percentile(diffs, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return {"point": float(point), "ci_low": float(lo), "ci_high": float(hi), "n_boot": n_boot}


def auroc_safe(scores, positive):
    """AUROC of `scores` for detecting `positive` events; NaN when only one class is present."""
    positive = np.asarray(positive).astype(int)
    if positive.min() == positive.max():
        return float("nan")
    return float(roc_auc_score(positive, scores))
'''

module_path = os.path.join(BASE_DIR, "meld_eval.py")
with open(module_path, "w") as f:
    f.write(MODULE_SRC)
import meld_eval
importlib.reload(meld_eval)
from meld_eval import (mean_std, paired_stats, holm, describe_diff, confusion_flat, macro_f1_from_conf, f1_per_class_from_conf,
                       dialogue_bootstrap_diff, auroc_safe)
from meld_models import load_split_dataset
from meld_context import make_loader
from meld_train import load_best_model, evaluate_model, save_predictions, weight_diagnostics
print(f"Wrote and imported {module_path}")

In [ ]:
test_df, test_ds = load_split_dataset(BASE_DIR, "test", IN_DIMS, EMOTIONS, context_config["speaker_mode"])
dev_df, dev_ds = load_split_dataset(BASE_DIR, "dev", IN_DIMS, EMOTIONS, context_config["speaker_mode"])
test_loader = make_loader(test_ds, batch_size=32, shuffle=False, seed=0)
val_loader = make_loader(dev_ds, batch_size=32, shuffle=False, seed=0)
N_TEST = len(test_df)

def jsonable(o):
    if isinstance(o, dict):
        return {k: jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    return o

def flat(m):
    d = {"accuracy": m["accuracy"], "weighted_f1": m["weighted_f1"], "macro_f1": m["macro_f1"]}
    d.update({f"f1_{e}": m["per_class"][e]["f1"] for e in EMOTIONS})
    d["minority_f1"] = float(np.mean([m["per_class"]["fear"]["f1"], m["per_class"]["disgust"]["f1"]]))
    return d

records, integrity = [], []
for rec in run_index:
    run_dir = rec["run_dir"]
    model, ck = load_best_model(run_dir, device)
    cfg = json.load(open(os.path.join(run_dir, "config.json")))
    res = json.load(open(os.path.join(run_dir, "result.json")))
    assert ck["fingerprint"] == cfg["fingerprint"], f"{run_dir}: checkpoint does not belong to this run configuration"
    vm, _ = evaluate_model(model, val_loader, device, EMOTIONS, eval_seed=0)
    assert abs(vm["macro_f1"] - res["best_val_macro_f1"]) < 1e-6, f"{run_dir}: checkpoint does not reproduce its recorded val macro-F1"
    integrity.append(abs(vm["macro_f1"] - res["best_val_macro_f1"]))

    tm, tarr = evaluate_model(model, test_loader, device, EMOTIONS, eval_seed=0)
    out_dir = os.path.join(P10_DIR, rec["name"], f"seed{rec['seed']}")
    os.makedirs(out_dir, exist_ok=True)
    tm["weights"] = weight_diagnostics(tarr, model.modalities)
    tm["val_macro_f1_of_checkpoint"] = vm["macro_f1"]
    tm["best_epoch"] = ck["epoch"]
    json.dump(jsonable(tm), open(os.path.join(out_dir, "metrics.json"), "w"), indent=2)
    save_predictions(os.path.join(out_dir, "test_predictions.csv"), tarr, EMOTIONS, model.modalities)
    records.append({"name": rec["name"], "seed": rec["seed"], "dir": out_dir, **flat(tm)})
    print(f"{rec['name']:18s} seed {rec['seed']}: test acc {tm['accuracy']:.4f} | wF1 {tm['weighted_f1']:.4f} | macro-F1 {tm['macro_f1']:.4f} | "
          f"fear F1 {tm['per_class']['fear']['f1']:.3f} | disgust F1 {tm['per_class']['disgust']['f1']:.3f}")
runs = pd.DataFrame(records)
runs.drop(columns="dir").to_csv(os.path.join(RESULTS_DIR, "phase10_test_runs.csv"), index=False)

In [ ]:
METRICS = ["accuracy", "weighted_f1", "macro_f1"] + [f"f1_{e}" for e in EMOTIONS] + ["minority_f1"]
agg = {}
for name, g in runs.groupby("name", sort=False):
    agg[name] = {k: dict(zip(("mean", "std"), mean_std(g[k].values))) for k in METRICS}
    agg[name]["n_seeds"] = int(len(g))

def fmt(name, k):
    return f"{agg[name][k]['mean']:.4f} +/- {agg[name][k]['std']:.4f}"

main = pd.DataFrame({
    "Metric": ["Accuracy", "Weighted F1", "Macro F1", "Fear F1 (minority)", "Disgust F1 (minority)", "Minority mean F1"],
    f"{BASELINE}": [fmt(BASELINE, k) for k in ("accuracy", "weighted_f1", "macro_f1", "f1_fear", "f1_disgust", "minority_f1")],
    f"{PROPOSED}": [fmt(PROPOSED, k) for k in ("accuracy", "weighted_f1", "macro_f1", "f1_fear", "f1_disgust", "minority_f1")],
})
print(f"=== TEST results, mean +/- std over {len(SEEDS)} seeds ===")
print(main.to_string(index=False))
main.to_csv(os.path.join(RESULTS_DIR, "phase10_main_comparison.csv"), index=False)

per_class = pd.DataFrame({n: {e: f"{agg[n]['f1_' + e]['mean']:.3f} +/- {agg[n]['f1_' + e]['std']:.3f}" for e in EMOTIONS} for n in (BASELINE, PROPOSED)})
print("\nPer-class F1:\n" + per_class.to_string())
per_class.to_csv(os.path.join(RESULTS_DIR, "phase10_per_class_f1.csv"))

all_exp = pd.DataFrame({n: {k: f"{agg[n][k]['mean']:.4f} +/- {agg[n][k]['std']:.4f}" for k in ("accuracy", "weighted_f1", "macro_f1", "minority_f1")} for n in agg}).T
print("\nAll experiments (descriptive; lambda was chosen on validation only):\n" + all_exp.to_string())
all_exp.to_csv(os.path.join(RESULTS_DIR, "phase10_all_experiments.csv"))

In [ ]:
pivot = {k: runs.pivot(index="seed", columns="name", values=k) for k in METRICS}
paired = {k: paired_stats(pivot[k][PROPOSED].values, pivot[k][BASELINE].values) for k in METRICS}
cls_keys = [f"f1_{e}" for e in EMOTIONS]
adj = holm([paired[k]["p_value"] for k in cls_keys])
for k, a in zip(cls_keys, adj):
    paired[k]["p_holm_over_classes"] = float(a)

rows = [{"metric": k, **{kk: paired[k][kk] for kk in ("mean_diff", "std_diff", "ci_low", "ci_high", "p_value", "n_a_higher", "n")},
         "p_holm_over_classes": paired[k].get("p_holm_over_classes", np.nan)} for k in METRICS]
paired_df = pd.DataFrame(rows)
print(f"Paired differences ({PROPOSED} - {BASELINE}) over seeds:")
print(paired_df.round(4).to_string(index=False))
paired_df.to_csv(os.path.join(RESULTS_DIR, "phase10_paired_stats.csv"), index=False)

dlg = test_df["Dialogue_ID"].values
boot = []
for sd in SEEDS:
    pb = pd.read_csv(os.path.join(P10_DIR, BASELINE, f"seed{sd}", "test_predictions.csv"))
    pp = pd.read_csv(os.path.join(P10_DIR, PROPOSED, f"seed{sd}", "test_predictions.csv"))
    b = dialogue_bootstrap_diff(pp["y_true"].values, pp["y_pred"].values, pb["y_pred"].values, dlg, C, n_boot=1000, seed=sd)
    boot.append({"seed": sd, **b, "ci_excludes_zero": bool(b["ci_low"] > 0 or b["ci_high"] < 0)})
boot_df = pd.DataFrame(boot)
print("\nPer-seed paired dialogue bootstrap of macro-F1 difference (95% percentile CI):")
print(boot_df.round(4).to_string(index=False))
boot_df.to_csv(os.path.join(RESULTS_DIR, "phase10_bootstrap.csv"), index=False)

In [ ]:
def summed_confusion(name):
    return sum(np.array(json.load(open(os.path.join(P10_DIR, name, f"seed{sd}", "metrics.json")))["confusion_matrix"]) for sd in SEEDS)

conf = {n: summed_confusion(n) for n in (BASELINE, PROPOSED)}
conf_norm = {n: conf[n] / conf[n].sum(1, keepdims=True) for n in conf}
for n in conf:
    pd.DataFrame(conf[n], index=EMOTIONS, columns=EMOTIONS).to_csv(os.path.join(RESULTS_DIR, f"phase10_confusion_{n}.csv"))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
panels = [(f"{BASELINE}", conf_norm[BASELINE], "Blues", (0, 1)), (f"{PROPOSED}", conf_norm[PROPOSED], "Blues", (0, 1)),
          (f"difference (proposed - baseline)", conf_norm[PROPOSED] - conf_norm[BASELINE], "coolwarm", (-0.2, 0.2))]
for ax, (title, M, cmap, (lo, hi)) in zip(axes, panels):
    im = ax.imshow(M, cmap=cmap, vmin=lo, vmax=hi)
    ax.set_xticks(range(C)); ax.set_yticks(range(C)); ax.set_xticklabels(EMOTIONS, rotation=45, ha="right"); ax.set_yticklabels(EMOTIONS)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)
    for i in range(C):
        for j in range(C):
            ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=7, color="black")
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(f"Row-normalised test confusion matrices (summed over {len(SEEDS)} seeds)")
fig.savefig(os.path.join(FIG_DIR, "confusion_matrices.png"), dpi=200, bbox_inches="tight"); plt.close(fig)
print("Saved confusion matrices.")

In [ ]:
manifest_test = pd.read_csv(os.path.join(BASE_DIR, "data", "context_manifest_test.csv"))
key2row = {(int(d), int(u)): int(r) for r, d, u in zip(manifest_test["row_idx"], manifest_test["Dialogue_ID"], manifest_test["Utterance_ID"])}
overlap_rows = sorted({key2row[(int(d), int(u))] for o in leakage["utterance_overlaps"] if "test" in o["splits"]
                       for d, u in o["test_dialogue_utterance"]})
keep = np.ones(N_TEST, dtype=bool); keep[overlap_rows] = False
from sklearn.metrics import f1_score
sens = {}
for n in (BASELINE, PROPOSED):
    vals = []
    for sd in SEEDS:
        p = pd.read_csv(os.path.join(P10_DIR, n, f"seed{sd}", "test_predictions.csv"))
        vals.append(f1_score(p["y_true"].values[keep], p["y_pred"].values[keep], average="macro", labels=list(range(C)), zero_division=0))
    sens[n] = dict(zip(("mean", "std"), mean_std(vals)))
print(f"Test rows duplicated from train (kept in the main results): {overlap_rows}")
print("Macro-F1 excluding those rows:", {n: f"{v['mean']:.4f} +/- {v['std']:.4f}" for n, v in sens.items()})

In [ ]:
results = []
def check(name, cond, detail=""):
    detail = str(detail)
    results.append({"test": name, "passed": bool(cond), "detail": detail})
    print(("PASS  " if cond else "FAIL  ") + name + (f"  [{detail}]" if detail else ""))

from sklearn.metrics import accuracy_score
worst, aligned, conf_ok = 0.0, True, True
for r in records:
    p = pd.read_csv(os.path.join(r["dir"], "test_predictions.csv"))
    m = json.load(open(os.path.join(r["dir"], "metrics.json")))
    y, yp = p["y_true"].values, p["y_pred"].values
    ind = {"accuracy": accuracy_score(y, yp), "weighted_f1": f1_score(y, yp, average="weighted", zero_division=0),
           "macro_f1": f1_score(y, yp, average="macro", labels=list(range(C)), zero_division=0)}
    ind.update({f"f1_{e}": f1_score(y, yp, labels=[i], average=None, zero_division=0)[0] for i, e in enumerate(EMOTIONS)})
    worst = max(worst, max(abs(ind[k] - r[k]) for k in ind))
    aligned &= bool(np.array_equal(p["row_idx"].values, np.arange(N_TEST)) and np.array_equal(y, test_ds.labels))
    cm = np.array(m["confusion_matrix"])
    conf_ok &= bool(cm.sum() == N_TEST and np.array_equal(cm.sum(1), np.bincount(y, minlength=C)))
check("E1 saved metrics equal an independent sklearn recomputation from the prediction files", worst < 1e-12, f"max |diff|={worst:.1e}")
check("E2 predictions cover every test row once, in CSV order, with the CSV labels", aligned)
check("E3 every checkpoint matches its run config and reproduces its recorded validation macro-F1", len(integrity) == len(run_index) and max(integrity) < 1e-6, f"max |diff|={max(integrity):.1e}")
check("E4 confusion matrices sum to the test size with correct row totals", conf_ok)

a, b = np.array([0.5, 0.6, 0.7]), np.array([0.4, 0.5, 0.5])
ps = paired_stats(a, b)
from scipy import stats as sps
ref = sps.ttest_rel(a, b)
ci = sps.t.interval(0.95, 2, loc=0.13333333, scale=0.05773503 / np.sqrt(3))
check("E5 paired_stats = scipy ttest_rel + t-interval", abs(ps["p_value"] - ref.pvalue) < 1e-12 and abs(ps["ci_low"] - ci[0]) < 1e-6 and abs(ps["mean_diff"] - 0.13333333) < 1e-6)
check("E6 Holm correction on a known example", np.allclose(holm([0.01, 0.04, 0.03]), [0.03, 0.06, 0.06]) and np.isnan(holm([0.01, np.nan])[1]))

rs = np.random.RandomState(0)
yt = rs.randint(0, C, 600); yp1 = np.where(rs.rand(600) < 0.5, yt, rs.randint(0, C, 600)); dl = np.repeat(np.arange(60), 10)
check("E7a fast macro-F1 (confusion-based) equals sklearn", abs(macro_f1_from_conf(confusion_flat(yt, yp1, C), C) - f1_score(yt, yp1, average="macro", labels=list(range(C)), zero_division=0)) < 1e-12)
same = dialogue_bootstrap_diff(yt, yp1, yp1, dl, C, n_boot=200, seed=0)
better = dialogue_bootstrap_diff(yt, yt, yp1, dl, C, n_boot=200, seed=0)
check("E7b bootstrap: identical predictions -> difference exactly 0; perfect vs noisy -> CI above 0",
      same["point"] == 0 and same["ci_low"] == 0 and same["ci_high"] == 0 and better["ci_low"] > 0)
check("E8 both compared models have the same seeds", set(runs[runs.name == BASELINE].seed) == set(runs[runs.name == PROPOSED].seed) == set(SEEDS))
check("E9 duplicated-utterance rows located in the test split", len(overlap_rows) == sum(len(o["test_dialogue_utterance"]) for o in leakage["utterance_overlaps"] if "test" in o["splits"]),
      f"{len(overlap_rows)} rows")

failed = [r for r in results if not r["passed"]]
with open(os.path.join(RESULTS_DIR, "evaluation_verification.json"), "w") as f:
    json.dump({"n_tests": len(results), "n_failed": len(failed), "results": results}, f, indent=2)
assert not failed, f"{len(failed)} verification test(s) failed: {[r['test'] for r in failed]}"
print(f"\nAll {len(results)} verification checks passed.")

In [ ]:
summary = {
    "baseline": BASELINE, "proposed": PROPOSED, "seeds": SEEDS, "n_test_utterances": N_TEST, "selected_lambda": selected,
    "aggregate": agg, "paired_proposed_minus_baseline": paired, "bootstrap_per_seed": boot,
    "confusion_summed_over_seeds": {n: conf[n] for n in conf},
    "sensitivity_excluding_train_duplicates": {"excluded_test_rows": overlap_rows, "macro_f1": sens},
    "checkpoint_selection": p9cfg["selection_metric"],
}
with open(os.path.join(RESULTS_DIR, "phase10_test_results.json"), "w") as f:
    json.dump(jsonable(summary), f, indent=2)
print("Saved results/phase10_test_results.json")
print("Phase 10 Evaluation completed successfully.")